Connected to raoxuan (Python 3.9.18)

In [ ]:
import torch
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor

# 导入项目模块
from src.classifiers.lr_rgda_classifier import LRRGDAClassifier, EnsembleClassifier
from src.detectors.ood_detector import ClassifierBasedOODDetector, build_stats_dict_from_features
from src.routing.adaptive_router import AdaptiveRouter

# 设置设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

使用设备: cuda


In [ ]:
ID_DATASETS = ['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers']
OOD_DATASETS = ['food101', 'mnist', 'oxford_pets']

CACHE_DIR = 'cache/pretrained_features'

print("=" * 80)
print("数据集配置")
print("=" * 80)
print(f"ID数据集: {ID_DATASETS}")
print(f"OOD数据集: {OOD_DATASETS}")
print(f"缓存目录: {CACHE_DIR}")

数据集配置
ID数据集: ['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers']
OOD数据集: ['food101', 'mnist', 'oxford_pets']
缓存目录: cache/pretrained_features


In [ ]:
# 获取logit_scale
logit_scale = model.logit_scale.exp().item()
print(f"    Logit scale: {logit_scale:.4f}")

# [4. 收集类别名称和特征]
print("\n[2/6] 收集类别名称...")

# 收集所有类别名（ID + OOD）
all_class_names = []
dataset_info = {}

for ds in ID_DATASETS + OOD_DATASETS:
    with open(f'{CACHE_DIR}/{ds}_features.pkl', 'rb') as f:
        data = pickle.load(f)
    
    num_classes = len(data['class_names'])
    dataset_info[ds] = {
        'num_classes': num_classes,
        'train_samples': len(data['train_features']),
        'test_samples': len(data['test_features']),
        'class_names': data['class_names']
    }
    all_class_names.extend(data['class_names'])

total_classes = len(all_class_names)
num_id_classes = sum(dataset_info[ds]['num_classes'] for ds in ID_DATASETS)

print(f"    总类别数: {total_classes}")
print(f"    ID类别数: {num_id_classes}")
print(f"    OOD类别数: {total_classes - num_id_classes}")

print("\n数据集详情:")
for ds, info in dataset_info.items():
    ds_type = "ID" if ds in ID_DATASETS else "OOD"
    print(f"    {ds:15s} ({ds_type}): {info['num_classes']:3d} classes, "
          f"{info['train_samples']:4d} train, {info['test_samples']:4d} test")

NameError: name 'model' is not defined

In [ ]:
print("\n[1/6] 加载CLIP模型...")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
model.eval()


[1/6] 加载CLIP模型...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
# 获取logit_scale
logit_scale = model.logit_scale.exp().item()
print(f"    Logit scale: {logit_scale:.4f}")

# [4. 收集类别名称和特征]
print("\n[2/6] 收集类别名称...")

# 收集所有类别名（ID + OOD）
all_class_names = []
dataset_info = {}

for ds in ID_DATASETS + OOD_DATASETS:
    with open(f'{CACHE_DIR}/{ds}_features.pkl', 'rb') as f:
        data = pickle.load(f)
    
    num_classes = len(data['class_names'])
    dataset_info[ds] = {
        'num_classes': num_classes,
        'train_samples': len(data['train_features']),
        'test_samples': len(data['test_features']),
        'class_names': data['class_names']
    }
    all_class_names.extend(data['class_names'])

total_classes = len(all_class_names)
num_id_classes = sum(dataset_info[ds]['num_classes'] for ds in ID_DATASETS)

print(f"    总类别数: {total_classes}")
print(f"    ID类别数: {num_id_classes}")
print(f"    OOD类别数: {total_classes - num_id_classes}")

print("\n数据集详情:")
for ds, info in dataset_info.items():
    ds_type = "ID" if ds in ID_DATASETS else "OOD"
    print(f"    {ds:15s} ({ds_type}): {info['num_classes']:3d} classes, "
          f"{info['train_samples']:4d} train, {info['test_samples']:4d} test")

    Logit scale: 100.0000

[2/6] 收集类别名称...
    总类别数: 507
    ID类别数: 359
    OOD类别数: 148

数据集详情:
    aircraft        (ID): 100 classes, 1600 train, 1000 test
    caltech101      (ID): 100 classes, 1600 train, 1000 test
    dtd             (ID):  47 classes,  752 train, 1000 test
    eurosat         (ID):  10 classes,  160 train, 1000 test
    flowers         (ID): 102 classes, 1632 train, 1000 test
    food101         (OOD): 101 classes, 1616 train, 1000 test
    mnist           (OOD):  10 classes,  160 train, 1000 test
    oxford_pets     (OOD):  37 classes,  592 train, 1000 test


In [ ]:
logit_scale

100.0

In [ ]:
dataset_info.keys()


dict_keys(['aircraft', 'caltech101', 'dtd', 'eurosat', 'flowers', 'food101', 'mnist', 'oxford_pets'])

In [ ]:
# [5. 加载ID训练特征并构建统计分布]
print("\n[3/6] 加载ID训练特征并构建统计分布...")

all_train_features = []
all_train_labels = []
label_offset = 0

for ds in ID_DATASETS:
    with open(f'{CACHE_DIR}/{ds}_features.pkl', 'rb') as f:
        data = pickle.load(f)
    
    all_train_features.append(data['train_features'])
    all_train_labels.append(data['train_labels'] + label_offset)
    label_offset += len(data['class_names'])

all_train_features = torch.cat(all_train_features)
all_train_labels = torch.cat(all_train_labels)

print(f"    总训练样本: {len(all_train_features)}")
print(f"    训练特征维度: {all_train_features.shape}")

# 构建统计分布
print("\n构建stats_dict...")
stats_dict = build_stats_dict_from_features(all_train_features, all_train_labels)
print(f"    Stats dict类别数: {len(stats_dict)}")


[3/6] 加载ID训练特征并构建统计分布...
    总训练样本: 5744
    训练特征维度: torch.Size([5744, 512])

构建stats_dict...
    Stats dict类别数: 359


In [ ]:
all_train_features

tensor([[ 0.0077, -0.0876,  0.0477,  ...,  0.0207,  0.0233,  0.0305],
        [ 0.0262, -0.0910,  0.0618,  ...,  0.0469,  0.0083,  0.0500],
        [-0.0060, -0.0543,  0.0266,  ..., -0.0002,  0.0043,  0.0331],
        ...,
        [-0.0290, -0.1063,  0.0064,  ...,  0.0171,  0.0055,  0.0342],
        [-0.0243, -0.0923,  0.0132,  ..., -0.0042,  0.0479,  0.0338],
        [-0.0522, -0.0816,  0.0222,  ..., -0.0200,  0.0036, -0.0104]])

In [ ]:
all_train_features.shape

torch.Size([5744, 512])

In [ ]:
# [6. 构建零样本分类器]
print("\n[4/6] 构建零样本分类器...")

zeroshot_weights = []
templates = [lambda x: f"a photo of a {x}."]

with torch.no_grad():
    for classname in tqdm(all_class_names, desc="Building zero-shot classifier"):
        classname = classname.replace('_', ' ')
        texts = [template(classname) for template in templates]
        text_inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
        class_embeddings = model.get_text_features(**text_inputs)
        class_embeddings = class_embeddings / class_embeddings.norm(dim=-1, keepdim=True)
        class_embedding = class_embeddings.mean(dim=0)
        class_embedding /= class_embedding.norm()
        zeroshot_weights.append(class_embedding)


[4/6] 构建零样本分类器...


Building zero-shot classifier: 100%|██████████| 507/507 [00:03<00:00, 131.76it/s]


In [ ]:
zeroshot_classifier = torch.stack(zeroshot_weights, dim=1).to(device)
print(f"    零样本分类器形状: {zeroshot_classifier.shape}")

    零样本分类器形状: torch.Size([512, 507])


In [ ]:
rint("\n[5/6] 构建LR-RGDA分类器和OOD检测器...")

# 构建LR-RGDA分类器
lr_rgda_classifier = LRRGDAClassifier(
    stats_dict=stats_dict,
    device=device,
    rank=32,
    qda_reg_alpha1=0.3,
    qda_reg_alpha2=2.0,
    qda_reg_alpha3=0.5,
    temperature=1.0
)

# 构建OOD检测器
ood_detector = ClassifierBasedOODDetector(
    stats_dict=stats_dict,
    classifier_type='lr_rgda',
    device=device,
    rank=32,
    qda_reg_alpha1=0.3,
    qda_reg_alpha2=2.0,
    qda_reg_alpha3=0.5
)

print("    ✓ LR-RGDA分类器构建完成")
print("    ✓ OOD检测器构建完成")

NameError: name 'rint' is not defined

In [ ]:
print("\n[5/6] 构建LR-RGDA分类器和OOD检测器...")

# 构建LR-RGDA分类器
lr_rgda_classifier = LRRGDAClassifier(
    stats_dict=stats_dict,
    device=device,
    rank=32,
    qda_reg_alpha1=0.3,
    qda_reg_alpha2=2.0,
    qda_reg_alpha3=0.5,
    temperature=1.0
)

# 构建OOD检测器
ood_detector = ClassifierBasedOODDetector(
    stats_dict=stats_dict,
    classifier_type='lr_rgda',
    device=device,
    rank=32,
    qda_reg_alpha1=0.3,
    qda_reg_alpha2=2.0,
    qda_reg_alpha3=0.5
)

print("    ✓ LR-RGDA分类器构建完成")
print("    ✓ OOD检测器构建完成")


[5/6] 构建LR-RGDA分类器和OOD检测器...
✓ LR_RGDA OOD Detector initialized with 359 classes
    ✓ LR-RGDA分类器构建完成
    ✓ OOD检测器构建完成


In [ ]:
# [8. 辅助函数定义]
def evaluate_classifier(classifier_fn, name, test_features, test_labels, is_id=True):
    """
    评估分类器性能
    classifier_fn: 预测函数，输入features，输出predictions
    """
    with torch.no_grad():
        predictions = classifier_fn(test_features)
    
    correct = (predictions.cpu() == test_labels.cpu()).sum().item()
    total = len(test_labels)
    accuracy = correct / total * 100
    
    ds_type = "ID" if is_id else "OOD"
    print(f"    {name:20s} ({ds_type}): {accuracy:5.1f}% ({correct}/{total})")
    return accuracy, correct, total


def get_test_data(dataset_list, is_id=True):
    """
    加载测试数据并应用正确的label offset
    """
    all_features = []
    all_labels = []
    
    # 计算基础offset
    if is_id:
        # ID数据集的offset从0开始
        base_offset = 0
        offset = 0
        for ds in ID_DATASETS:
            if ds in dataset_list:
                break
            offset += dataset_info[ds]['num_classes']
    else:
        # OOD数据集的offset从num_id_classes开始
        base_offset = num_id_classes
        offset = base_offset
        for ds in OOD_DATASETS:
            if ds in dataset_list:
                break
            offset += dataset_info[ds]['num_classes']
    
    for ds in dataset_list:
        with open(f'{CACHE_DIR}/{ds}_features.pkl', 'rb') as f:
            data = pickle.load(f)
        
        all_features.append(data['test_features'])
        all_labels.append(data['test_labels'] + offset)
        offset += dataset_info[ds]['num_classes']
    
    features = torch.cat(all_features).to(device)
    labels = torch.cat(all_labels).to(device)

In [ ]:
# [8. 辅助函数定义]
def evaluate_classifier(classifier_fn, name, test_features, test_labels, is_id=True):
    """
    评估分类器性能
    classifier_fn: 预测函数，输入features，输出predictions
    """
    with torch.no_grad():
        predictions = classifier_fn(test_features)
    
    correct = (predictions.cpu() == test_labels.cpu()).sum().item()
    total = len(test_labels)
    accuracy = correct / total * 100
    
    ds_type = "ID" if is_id else "OOD"
    print(f"    {name:20s} ({ds_type}): {accuracy:5.1f}% ({correct}/{total})")
    return accuracy, correct, total


def get_test_data(dataset_list, is_id=True):
    """
    加载测试数据并应用正确的label offset
    """
    all_features = []
    all_labels = []
    
    # 计算基础offset
    if is_id:
        # ID数据集的offset从0开始
        base_offset = 0
        offset = 0
        for ds in ID_DATASETS:
            if ds in dataset_list:
                break
            offset += dataset_info[ds]['num_classes']
    else:
        # OOD数据集的offset从num_id_classes开始
        base_offset = num_id_classes
        offset = base_offset
        for ds in OOD_DATASETS:
            if ds in dataset_list:
                break
            offset += dataset_info[ds]['num_classes']
    
    for ds in dataset_list:
        with open(f'{CACHE_DIR}/{ds}_features.pkl', 'rb') as f:
            data = pickle.load(f)
        
        all_features.append(data['test_features'])
        all_labels.append(data['test_labels'] + offset)
        offset += dataset_info[ds]['num_classes']
    
    features = torch.cat(all_features).to(device)
    labels = torch.cat(all_labels).to(device)

    return features, labels

In [ ]:
print("\n[6/6] 加载测试数据...")
id_test_features, id_test_labels = get_test_data(ID_DATASETS, is_id=True)
ood_test_features, ood_test_labels = get_test_data(OOD_DATASETS, is_id=False)

print(f"    ID测试样本: {len(id_test_features)}")
print(f"    OOD测试样本: {len(ood_test_features)}")


[6/6] 加载测试数据...
    ID测试样本: 5000
    OOD测试样本: 3000


In [ ]:
id_test_labels

tensor([ 64,  38,  30,  ..., 321, 297, 344], device='cuda:0')

In [ ]:
id_test_labels.min

<function Tensor.min>

In [ ]:
id_test_labels.min()

tensor(0, device='cuda:0')

In [ ]:
id_test_labels.min()

tensor(0, device='cuda:0')

In [ ]:
id_test_labels.max()

tensor(358, device='cuda:0')

In [ ]:
ood_test_labels.min()

tensor(359, device='cuda:0')

In [ ]:
# [9. 测试1: 纯零样本分类器]
print("\n" + "=" * 80)
print("测试1: 纯零样本分类器 (Zero-shot Only)")
print("=" * 80)

def zeroshot_predict(features):
    with torch.no_grad():
        logits = logit_scale * (features @ zeroshot_classifier)
        return logits.argmax(dim=1)

zeroshot_id_acc, zeroshot_id_correct, zeroshot_id_total = evaluate_classifier(
    zeroshot_predict, "Zero-shot", id_test_features, id_test_labels, is_id=True
)
zeroshot_ood_acc, zeroshot_ood_correct, zeroshot_ood_total = evaluate_classifier(
    zeroshot_predict, "Zero-shot", ood_test_features, ood_test_labels, is_id=False
)

zeroshot_overall_acc = (zeroshot_id_correct + zeroshot_ood_correct) / (zeroshot_id_total + zeroshot_ood_total) * 100
print(f"\n    平均准确率: {zeroshot_overall_acc:.1f}%")


测试1: 纯零样本分类器 (Zero-shot Only)
    Zero-shot            (ID):  46.7% (2337/5000)
    Zero-shot            (OOD):  71.4% (2141/3000)

    平均准确率: 56.0%


In [ ]:
zeroshot_logits = logit_scale * (id_test_features @ zeroshot_classifier)

In [ ]:
print()

In [ ]:
print()

In [ ]:
print()

In [ ]:
print(zeroshot_logits)

tensor([[24.1265, 23.0285, 27.3669,  ..., 16.9415, 15.3562, 17.0632],
        [28.9210, 27.6885, 30.6853,  ..., 16.4100, 19.3621, 16.1740],
        [28.2480, 24.3703, 27.6306,  ..., 14.7110, 15.8546, 12.5794],
        ...,
        [19.4004, 12.1067, 11.4076,  ..., 11.7356, 13.3150, 13.3957],
        [22.0893, 15.9913, 15.9507,  ..., 18.7287, 20.4968, 16.7704],
        [19.1436, 12.7925, 11.5457,  ...,  9.9087, 14.0471, 20.2764]],
       device='cuda:0')


In [ ]:
print(zeroshot_logits.norm(dim=-1))

tensor([377.5208, 429.6461, 377.9474,  ..., 411.2311, 449.5696, 406.4253],
       device='cuda:0')


In [ ]:
rgda_logits = lr_rgda_classifier(id_test_features)

TypeError: 'LRRGDAClassifier' object is not callable

In [ ]:
rgda_logits = lr_rgda_classifier.forward(id_test_features)

In [ ]:
rgda_logits.norm(dim=-1)

tensor([3246.8357, 3246.7563, 3246.2280,  ..., 3249.2769, 3250.9429,
        3249.8523], device='cuda:0')

In [ ]:
rgda_logits

tensor([[171.6426, 171.6688, 171.7725,  ..., 171.1540, 171.1622, 171.1753],
        [171.7662, 171.7888, 171.8482,  ..., 171.0961, 171.1492, 171.1117],
        [171.6954, 171.6879, 171.7268,  ..., 171.0807, 171.1576, 171.1066],
        ...,
        [171.2536, 171.2335, 171.2092,  ..., 171.8645, 171.8134, 171.8184],
        [171.2929, 171.2886, 171.2773,  ..., 171.8579, 171.8025, 171.7782],
        [171.2473, 171.2525, 171.2254,  ..., 171.7549, 171.8447, 171.8512]],
       device='cuda:0')

In [ ]:
from classifiers.gaussian_statistics import LRRGDA

ImportError: cannot import name 'LRRGDA' from 'classifiers.gaussian_statistics' (/home/raoxuan/projects/clip_ood/classifiers/gaussian_statistics.py)

In [ ]:
from classifiers.gaussian_classifier import LRRGDA

In [ ]:
lrrgda = LRRGDA(    stats_dict=stats_dict,
    device=device,
    rank=32,
    qda_reg_alpha1=0.3,
    qda_reg_alpha2=2.0,
    qda_reg_alpha3=0.5,
    temperature=1.0)



In [ ]:
x = id_test_features

In [ ]:
self = lrrgda

In [ ]:
import torch.nn.functional as F

In [ ]:
affine_logits = F.linear(x, self.affine_weights, self.affine_biases)

In [ ]:
affine_logits

tensor([[171.6410, 171.6673, 171.7713,  ..., 171.1535, 171.1619, 171.1750],
        [171.7651, 171.7881, 171.8479,  ..., 171.0954, 171.1487, 171.1114],
        [171.6931, 171.6865, 171.7258,  ..., 171.0800, 171.1574, 171.1062],
        ...,
        [171.2517, 171.2297, 171.2079,  ..., 171.8643, 171.8132, 171.8180],
        [171.2902, 171.2842, 171.2758,  ..., 171.8568, 171.8019, 171.7776],
        [171.2459, 171.2498, 171.2244,  ..., 171.7546, 171.8445, 171.8508]],
       device='cuda:0')

In [ ]:
self.affine_weights

tensor([[-0.0684, -0.1300,  0.0772,  ...,  0.0947,  0.0528,  0.0653],
        [-0.0395, -0.1349,  0.0890,  ...,  0.0809,  0.0742,  0.0959],
        [-0.0231, -0.1385,  0.0832,  ...,  0.0709,  0.0523,  0.0814],
        ...,
        [-0.0849, -0.1888,  0.0234,  ...,  0.0018,  0.0340,  0.0524],
        [-0.0201, -0.2000,  0.0533,  ...,  0.0075,  0.0633,  0.0053],
        [-0.0697, -0.1966,  0.0283,  ..., -0.0132,  0.0336,  0.0244]],
       device='cuda:0')

In [ ]:
affine_biases

NameError: name 'affine_biases' is not defined

In [ ]:
self.affine_biases

tensor([170.3760, 170.3605, 170.3548, 170.3615, 170.3653, 170.3362, 170.3330,
        170.3454, 170.3449, 170.3326, 170.3462, 170.3183, 170.3233, 170.3346,
        170.3638, 170.3239, 170.3475, 170.3571, 170.3172, 170.3312, 170.3167,
        170.3547, 170.3494, 170.3235, 170.3586, 170.3409, 170.3652, 170.3381,
        170.3372, 170.3456, 170.3404, 170.3068, 170.3240, 170.2900, 170.3332,
        170.3246, 170.3143, 170.3169, 170.3295, 170.2802, 170.3274, 170.3532,
        170.3322, 170.3243, 170.3629, 170.3253, 170.3199, 170.3047, 170.3341,
        170.2564, 170.2777, 170.3061, 170.3522, 170.3346, 170.3178, 170.3374,
        170.3661, 170.2916, 170.2921, 170.3479, 170.3512, 170.3392, 170.2875,
        170.3374, 170.3264, 170.3344, 170.3370, 170.3502, 170.3006, 170.3382,
        170.2766, 170.3064, 170.3002, 170.2883, 170.2750, 170.3011, 170.3525,
        170.3297, 170.3255, 170.2942, 170.2787, 170.3051, 170.3071, 170.2948,
        170.3603, 170.3528, 170.3584, 170.3314, 170.3331, 170.32

In [ ]:
affine_logits = F.linear(x, self.affine_weights)

In [ ]:
affine_logits

tensor([[1.2650, 1.3067, 1.4166,  ..., 0.8995, 0.8947, 0.9184],
        [1.3891, 1.4276, 1.4931,  ..., 0.8414, 0.8815, 0.8548],
        [1.3172, 1.3260, 1.3710,  ..., 0.8259, 0.8901, 0.8496],
        ...,
        [0.8757, 0.8692, 0.8532,  ..., 1.6103, 1.5460, 1.5614],
        [0.9142, 0.9237, 0.9211,  ..., 1.6028, 1.5346, 1.5210],
        [0.8699, 0.8892, 0.8696,  ..., 1.5006, 1.5773, 1.5943]],
       device='cuda:0')

In [ ]:
        U_term = torch.einsum('crd,bd->bcr', self.U_eff_T_B_inv, x)
        u_c = U_term - self.U_eff_T_B_inv_mu.unsqueeze(0) # [B, C, r]
        
        # 计算 u_c^T M^{-1} u_c
        # M_invs: [C, r, r]
        # temp = M^{-1} u_c : [C, r, r] @ [B, C, r, 1] -> [B, C, r]
        # 但 batch matmul 需要对齐:
        # u_c.unsqueeze(2): [B, C, 1, r]
        # M_invs: [C, r, r] -> 广播成 [B, C, r, r] 太大
        
        # 使用 einsum 高效计算: u_c [B, C, r], M [C, r, k] -> [B, C, k]
        M_u = torch.einsum('bcr,crk->bck', u_c, self.M_invs)
        
        # 点积求和
        quadratic = 0.5 * (u_c * M_u).sum(dim=-1) # [B, C]

In [ ]:
quadratic

tensor([[0.0016, 0.0015, 0.0012,  ..., 0.0005, 0.0003, 0.0003],
        [0.0012, 0.0006, 0.0003,  ..., 0.0006, 0.0004, 0.0004],
        [0.0022, 0.0014, 0.0010,  ..., 0.0007, 0.0002, 0.0004],
        ...,
        [0.0019, 0.0038, 0.0013,  ..., 0.0002, 0.0002, 0.0004],
        [0.0027, 0.0044, 0.0015,  ..., 0.0010, 0.0006, 0.0006],
        [0.0014, 0.0028, 0.0011,  ..., 0.0003, 0.0002, 0.0003]],
       device='cuda:0')

In [ ]:
rgda_logits = affine_logits + quadratic

In [ ]:
rgda_preds = rgda_logits.argmax(dim=-1)

In [ ]:
corrects = rgda_preds == id_test_labels

In [ ]:
acc = corrects.sum() / len(corrects)

In [ ]:
acc

tensor(0.6300, device='cuda:0')

In [ ]:
affine_logits = F.linear(x, self.affine_weights, self.affine_biases)

In [ ]:
rgda_logits = affine_logits + quadratic

In [ ]:
rgda_preds = rgda_logits.argmax(dim=-1)

In [ ]:
corrects = rgda_preds == id_test_labels

In [ ]:
acc = corrects.sum() / len(corrects)

In [ ]:
acc

tensor(0.6876, device='cuda:0')

In [ ]:
rgda_logits

tensor([[171.6426, 171.6688, 171.7725,  ..., 171.1540, 171.1622, 171.1753],
        [171.7662, 171.7888, 171.8482,  ..., 171.0961, 171.1492, 171.1117],
        [171.6954, 171.6879, 171.7268,  ..., 171.0807, 171.1576, 171.1066],
        ...,
        [171.2536, 171.2335, 171.2092,  ..., 171.8645, 171.8134, 171.8184],
        [171.2929, 171.2886, 171.2773,  ..., 171.8579, 171.8025, 171.7782],
        [171.2473, 171.2525, 171.2254,  ..., 171.7549, 171.8447, 171.8512]],
       device='cuda:0')

In [ ]:
F.softmax(rgda_logits, dim=-1)

tensor([[0.0036, 0.0037, 0.0041,  ..., 0.0022, 0.0022, 0.0022],
        [0.0041, 0.0041, 0.0044,  ..., 0.0021, 0.0022, 0.0021],
        [0.0039, 0.0039, 0.0040,  ..., 0.0021, 0.0023, 0.0022],
        ...,
        [0.0021, 0.0021, 0.0020,  ..., 0.0039, 0.0037, 0.0038],
        [0.0020, 0.0020, 0.0020,  ..., 0.0036, 0.0034, 0.0033],
        [0.0021, 0.0021, 0.0020,  ..., 0.0034, 0.0038, 0.0038]],
       device='cuda:0')

In [ ]:
F.softmax(rgda_logits - rgda_logits.max(dim=-1), dim=-1)

TypeError: unsupported operand type(s) for -: 'Tensor' and 'torch.return_types.max'

In [ ]:
rgda_logits - rgda_logits.max(-1)

TypeError: unsupported operand type(s) for -: 'Tensor' and 'torch.return_types.max'

In [ ]:
rgda_logits.max(-1)

torch.return_types.max(
values=tensor([171.8578, 171.8890, 171.9102,  ..., 172.0981, 172.0025, 171.9707],
       device='cuda:0'),
indices=tensor([ 65,  37,  32,  ..., 321, 297, 295], device='cuda:0'))

In [ ]:
F.softmax(rgda_logits - rgda_logits.max(dim=-1, keepdim=True), dim=-1)

TypeError: unsupported operand type(s) for -: 'Tensor' and 'torch.return_types.max'

In [ ]:
rgda_logits.max(dim=-1, keepdim=True)

torch.return_types.max(
values=tensor([[171.8578],
        [171.8890],
        [171.9102],
        ...,
        [172.0981],
        [172.0025],
        [171.9707]], device='cuda:0'),
indices=tensor([[ 65],
        [ 37],
        [ 32],
        ...,
        [321],
        [297],
        [295]], device='cuda:0'))

In [ ]:
F.softmax(rgda_logits - rgda_logits.max(dim=-1, keepdim=True).values, dim=-1)

tensor([[0.0036, 0.0037, 0.0041,  ..., 0.0022, 0.0022, 0.0022],
        [0.0041, 0.0041, 0.0044,  ..., 0.0021, 0.0022, 0.0021],
        [0.0039, 0.0039, 0.0040,  ..., 0.0021, 0.0023, 0.0022],
        ...,
        [0.0021, 0.0021, 0.0020,  ..., 0.0039, 0.0037, 0.0038],
        [0.0020, 0.0020, 0.0020,  ..., 0.0036, 0.0034, 0.0033],
        [0.0021, 0.0021, 0.0020,  ..., 0.0034, 0.0038, 0.0038]],
       device='cuda:0')

In [ ]:
F.softmax((rgda_logits - rgda_logits.max(dim=-1, keepdim=True).values) / 0.05, dim=-1)

tensor([[7.0797e-04, 1.1959e-03, 9.5212e-03,  ..., 4.0384e-08, 4.7590e-08,
         6.1834e-08],
        [6.7431e-03, 1.0590e-02, 3.4752e-02,  ..., 1.0183e-08, 2.9460e-08,
         1.3931e-08],
        [1.0899e-03, 9.3936e-04, 2.0436e-03,  ..., 4.9953e-09, 2.3249e-08,
         8.3870e-09],
        ...,
        [3.1283e-08, 2.0910e-08, 1.2872e-08,  ..., 6.3376e-03, 2.2779e-03,
         2.5169e-03],
        [9.5001e-08, 8.7167e-08, 6.9568e-08,  ..., 7.6762e-03, 2.5361e-03,
         1.5607e-03],
        [5.3883e-08, 5.9902e-08, 3.4828e-08,  ..., 1.3833e-03, 8.3350e-03,
         9.4864e-03]], device='cuda:0')

In [ ]:
shifted_rgda_logits = rgda_logits - rgda_logits.max(dim=-1, keepdim=True).values

In [ ]:
shifted_rgda_logits

tensor([[-0.2152, -0.1890, -0.0853,  ..., -0.7038, -0.6956, -0.6825],
        [-0.1227, -0.1002, -0.0408,  ..., -0.7929, -0.7398, -0.7772],
        [-0.2148, -0.2222, -0.1834,  ..., -0.8295, -0.7526, -0.8035],
        ...,
        [-0.8445, -0.8647, -0.8889,  ..., -0.2336, -0.2847, -0.2798],
        [-0.7097, -0.7140, -0.7253,  ..., -0.1447, -0.2001, -0.2243],
        [-0.7234, -0.7181, -0.7452,  ..., -0.2158, -0.1260, -0.1195]],
       device='cuda:0')

In [ ]:
zeroshot_logits = id_test_features @ zeroshot_classifier

In [ ]:
zeroshot_logits = zeroshot_logits - zeroshot_logits.max(dim=-1, keepdim=True)

TypeError: unsupported operand type(s) for -: 'Tensor' and 'torch.return_types.max'

In [ ]:
zeroshot_logits = zeroshot_logits - zeroshot_logits.max(dim=-1, keepdim=True).values

In [ ]:
zeroshot_logits

tensor([[-0.0427, -0.0537, -0.0103,  ..., -0.1145, -0.1304, -0.1133],
        [-0.0392, -0.0515, -0.0216,  ..., -0.1643, -0.1348, -0.1667],
        [-0.0264, -0.0652, -0.0326,  ..., -0.1618, -0.1504, -0.1831],
        ...,
        [-0.1422, -0.2151, -0.2221,  ..., -0.2188, -0.2030, -0.2022],
        [-0.0929, -0.1539, -0.1543,  ..., -0.1265, -0.1089, -0.1461],
        [-0.1506, -0.2141, -0.2266,  ..., -0.2430, -0.2016, -0.1393]],
       device='cuda:0')

In [ ]:
shifted_rgda_logits

tensor([[-0.2152, -0.1890, -0.0853,  ..., -0.7038, -0.6956, -0.6825],
        [-0.1227, -0.1002, -0.0408,  ..., -0.7929, -0.7398, -0.7772],
        [-0.2148, -0.2222, -0.1834,  ..., -0.8295, -0.7526, -0.8035],
        ...,
        [-0.8445, -0.8647, -0.8889,  ..., -0.2336, -0.2847, -0.2798],
        [-0.7097, -0.7140, -0.7253,  ..., -0.1447, -0.2001, -0.2243],
        [-0.7234, -0.7181, -0.7452,  ..., -0.2158, -0.1260, -0.1195]],
       device='cuda:0')

In [ ]:
zeroshot_logits.norm(dim=-1)

tensor([2.9339, 3.4699, 3.4982,  ..., 3.7002, 2.7102, 3.8193], device='cuda:0')

In [ ]:
shifted_rgda_logits.norm(dim=-1)

tensor([10.3847, 11.1478, 11.9832,  ..., 12.3184,  9.0104,  9.5701],
       device='cuda:0')

In [ ]:
(5 * shifted_rgda_logits).norm(dim=-1)

tensor([51.9235, 55.7388, 59.9158,  ..., 61.5922, 45.0518, 47.8507],
       device='cuda:0')

In [ ]:
(3 * shifted_rgda_logits).norm(dim=-1)

tensor([31.1541, 33.4433, 35.9495,  ..., 36.9553, 27.0311, 28.7104],
       device='cuda:0')

In [ ]:
(2 * shifted_rgda_logits).norm(dim=-1)

tensor([20.7694, 22.2955, 23.9663,  ..., 24.6369, 18.0207, 19.1403],
       device='cuda:0')

In [ ]:
(1 * shifted_rgda_logits).norm(dim=-1)

tensor([10.3847, 11.1478, 11.9832,  ..., 12.3184,  9.0104,  9.5701],
       device='cuda:0')

In [ ]:
(5 * zeroshot_logits).norm(dim=-1)

tensor([14.6695, 17.3497, 17.4909,  ..., 18.5011, 13.5511, 19.0966],
       device='cuda:0')

In [ ]:
(4 * zeroshot_logits).norm(dim=-1)

tensor([11.7356, 13.8798, 13.9927,  ..., 14.8009, 10.8409, 15.2773],
       device='cuda:0')

In [ ]:
(3 * zeroshot_logits).norm(dim=-1)

tensor([ 8.8017, 10.4098, 10.4945,  ..., 11.1007,  8.1307, 11.4580],
       device='cuda:0')